# TabPFN – Stage 2
Target: `default` (1 = loan defaulted, 0 = fully paid). Uses `processed/stage2_tfm.csv` with a train/test split only (done in PyTorch).

In [ ]:
# !pip install torch tabpfn pandas scikit-learn shap dice-ml shapiq matplotlib
import gc, os, getpass, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
import shap
import dice_ml
import shapiq
from tabpfn import TabPFNClassifier, TabPFNRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score, mean_absolute_error, classification_report

In [ ]:
# TabPFN reads TABPFN_TOKEN from the environment when it loads the weights.
# Set it here (prompted, so it never gets saved in the notebook) if the kernel doesn't have it.
if not os.environ.get("TABPFN_TOKEN"):
    os.environ["TABPFN_TOKEN"] = getpass.getpass("Prior Labs token: ")
print("Token set:", bool(os.environ.get("TABPFN_TOKEN")))

In [ ]:
# ---- Config ----
DATA_PATH = "processed/stage2_tfm.csv"
TARGET = "default"
TASK = "classification"  # "auto", "classification" or "regression"
TEST_SIZE = 0.25
SEED = 42
N_ESTIMATORS = 2        # ensemble size; fewer = less GPU memory (default 8)
PRED_BATCH = 100         # test rows per predict call; lower if CUDA runs out of memory
N_BACKGROUND = 10        # SHAP / shapiq: background rows used to "remove" features
N_EXPLAIN = 50          # SHAP: number of test rows to explain (cost grows linearly)
N_IQ_ROWS = 10           # shapiq: number of test rows to explain
N_CF_APPLICANTS = 3      # DiCE: number of test rows to find counterfactuals for
N_CFS_EACH = 2           # DiCE: counterfactuals per applicant
FLIP_FROM = 1           # DiCE: predicted class we try to flip (1 = default)
N_TOP = 10              # scatter plots and heat map: show only the N_TOP most important features
IQ_BUDGET = 512         # shapiq: feature subsets evaluated per row (64 = all 2^6 subsets, exact; with more features it is an estimate)
SHAP_MAX_EVALS = 300   # SHAP: model evaluations per row ("auto" = SHAP decides)
NET_THRESHOLD = 0.01       # network plot: hide pairs with |value| below this
CLASS_NAMES = ["fully paid", "default"]   # names of class 0 and class 1
CLASS_COLORS = ["tab:green", "tab:red"]   # plot colours of class 0 and class 1

# ---- Cache for the slow steps (SHAP, shapiq, DiCE) ----
# The first run computes and saves the result; later runs load it. Delete the `cache` folder to recompute.
CACHE_DIR = "cache"
def cached(name, compute):
    path = f"{CACHE_DIR}/{os.path.basename(DATA_PATH)[:-4]}_{name}_seed{SEED}_est{N_ESTIMATORS}.pkl"
    if os.path.exists(path):
        print("Loaded from cache:", path)
        with open(path, "rb") as f:
            return pickle.load(f)
    result = compute()
    os.makedirs(CACHE_DIR, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(result, f)
    return result

In [ ]:
# Load the merged CSV
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
# SHAP only works on numbers, so turn text columns (e.g. `state`) into integer codes once, here.
# TabPFN is told which columns are categorical, so it still treats them as categories.
cat_cols = list(df.drop(columns=[TARGET]).select_dtypes(include=["object", "bool", "category"]).columns)
category_names = {col: list(df[col].astype("category").cat.categories) for col in cat_cols}  # code -> name
for col in cat_cols:
    df[col] = df[col].astype("category").cat.codes
print("Categorical columns:", cat_cols)

X, y = df.drop(columns=[TARGET]), df[TARGET]
cat_idx = [list(X.columns).index(col) for col in cat_cols]   # positions of categorical columns

task = TASK
if task == "auto":
    task = "classification" if y.nunique() <= 20 else "regression"

# Train / test split only (PyTorch random permutation)
perm = torch.randperm(len(df), generator=torch.Generator().manual_seed(SEED)).numpy()
n_test = int(len(df) * TEST_SIZE)
test_idx, train_idx = perm[:n_test], perm[n_test:]
X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
print("Task:", task, "|", X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True) if task == "classification" else y_train.describe())

In [ ]:
# TabPFN is a PyTorch transformer; pick the device and seed torch explicitly
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()   # release memory cached by earlier runs in this kernel
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
print("torch", torch.__version__, "| device:", device)

In [ ]:
Model = TabPFNClassifier if task == "classification" else TabPFNRegressor
# fit_with_cache keeps the processed training rows on the GPU, so every predict call is fast
# (important for SHAP and DiCE, which call the model thousands of times)
model = Model(device=device, n_estimators=N_ESTIMATORS, categorical_features_indices=cat_idx, fit_mode="fit_with_cache")
model.fit(X_train, y_train)   # no gradient training, just stores data as context

In [ ]:
# Predict the test set in chunks so activations fit in GPU memory
chunks = [X_test.iloc[i:i + PRED_BATCH] for i in range(0, len(X_test), PRED_BATCH)]
with torch.no_grad():         # inference only
    if task == "classification":
        proba = np.vstack([model.predict_proba(ch) for ch in chunks])
        pred = model.classes_[proba.argmax(1)]
    else:
        pred = np.concatenate([model.predict(ch) for ch in chunks])

In [ ]:
if task == "classification":
    print("Accuracy:", accuracy_score(y_test, pred))
    if proba.shape[1] == 2:
        print("ROC AUC:", roc_auc_score(y_test, proba[:, 1]))
    # precision, recall, f1-score and support (number of test rows) for each class
    print(classification_report(y_test, pred, target_names=CLASS_NAMES, digits=3))
else:
    print("R2:", r2_score(y_test, pred))
    print("MAE:", mean_absolute_error(y_test, pred))

## Part 1: SHAP on TabPFN

`shap.DeepExplainer` does not work here: it needs a plain `torch.nn.Module` whose gradients it can follow layer by layer.
`TabPFNClassifier` is a wrapper that does its own preprocessing and feeds the whole training set through the transformer as context.

So we use the model-agnostic `shap.Explainer` (as in AP2_XAI). It only needs a **function** that takes data and returns a prediction.
We explain the probability of default (class 1).

In [ ]:
# Prediction function: data in -> probability of the positive class (TARGET = 1) out (predicted in chunks to save GPU memory)
def predict_positive(data):
    data = pd.DataFrame(data, columns=X_train.columns).astype(X_train.dtypes.to_dict())
    out = []
    for i in range(0, len(data), PRED_BATCH):
        out.append(model.predict_proba(data.iloc[i:i + PRED_BATCH])[:, 1])
    return np.concatenate(out)

def run_shap():
    # Small background set + a sample of test rows (SHAP cost grows with both)
    background = shap.sample(X_train, N_BACKGROUND, random_state=SEED)
    explainer = shap.Explainer(predict_positive, background)
    return explainer(X_test.iloc[:N_EXPLAIN], max_evals=SHAP_MAX_EVALS)

shap_values = cached(f"shap_bg{N_BACKGROUND}_rows{N_EXPLAIN}", run_shap)

### Plot 1.1 - Beeswarm global plot

In [ ]:
plt.figure(figsize=(10, 5))
shap.plots.beeswarm(shap_values, show=False)
plt.title("1. SHAP Summary Plot (Beeswarm)")
plt.tight_layout()
plt.show()

### Plot 1.2 - Waterfall plots (local): 4 different applicants
The applicant least likely to default, the one most likely to default, and the two closest to the 50/50 boundary. The SHAP values start at the base value (average prediction) and add up to the final P(default).

In [ ]:
pred_shap = shap_values.base_values + shap_values.values.sum(1)   # SHAP values add up to P(TARGET = 1)
closest_to_half = list(np.argsort(np.abs(pred_shap - 0.5))[:2])
picks = list(dict.fromkeys([np.argmin(pred_shap), np.argmax(pred_shap)] + closest_to_half))   # no duplicates

for i in picks:
    plt.figure(figsize=(8, 4))
    shap.plots.waterfall(shap_values[int(i)], show=False)
    plt.title(f"Test row {i}: P({TARGET}) = {pred_shap[i]:.2f}, true label = {y_test.iloc[i]}")
    plt.tight_layout()
    plt.show()

### Plot 1.3 - Scatter (dependence) plots: one per feature (the `N_TOP` most important)
Each dot is one applicant: x = feature value, y = SHAP value of that feature. The colour is the class the **model predicted** for that applicant (not the true label). Text features such as `grade` are shown as their integer code.

In [ ]:
# Features ordered from most to least important (average absolute SHAP value)
order = np.abs(shap_values.values).mean(0).argsort()[::-1]
pred_class = pred[:N_EXPLAIN]        # class predicted by the model for the explained rows

for i in order[:N_TOP]:
    feature = X_train.columns[i]
    plt.figure(figsize=(7, 4))
    for cls, color, name in [(1, CLASS_COLORS[1], f"predicted {CLASS_NAMES[1]}"), (0, CLASS_COLORS[0], f"predicted {CLASS_NAMES[0]}")]:
        rows_cls = pred_class == cls
        plt.scatter(shap_values.data[rows_cls, i], shap_values.values[rows_cls, i], c=color, s=25, alpha=0.8, label=name)
    plt.axhline(0, color="gray", linewidth=0.8)
    plt.xlabel(feature)
    plt.ylabel(f"SHAP value for {feature}")
    plt.title(f"SHAP scatter plot: {feature}")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Part 2: Feature interactions with shapiq

SHAP gives one number per feature. `shapiq` also gives numbers for **pairs of features** (order 2): *do two features together push the prediction more (or less) than the sum of their separate effects?*
We use the `k-SII` index with `max_order=2`. With 35 features there are 2^35 feature subsets, far too many to evaluate, so shapiq **estimates** the values from `IQ_BUDGET` evaluations per row. The numbers are therefore approximate (and the plots are noisier than for 6 features).

In [ ]:
# Same idea as before: a function from data to P(TARGET = 1). shapiq passes numpy arrays.
rows = X_test.iloc[:N_IQ_ROWS].values
feature_names = list(X_train.columns)

def run_shapiq():
    explainer_iq = shapiq.TabularExplainer(
        model=predict_positive,
        data=shap.sample(X_train, N_BACKGROUND, random_state=SEED).values,   # background sample
        index="k-SII",      # Shapley interaction index
        max_order=2,        # single features (order 1) and pairs of features (order 2)
        random_state=SEED,
    )
    # budget = how many feature subsets shapiq evaluates per row (see IQ_BUDGET in the config)
    return [explainer_iq.explain(x, budget=IQ_BUDGET) for x in rows]

interactions = cached(f"shapiq_bg{N_BACKGROUND}_rows{N_IQ_ROWS}", run_shapiq)
print(interactions[0])

### Plot 2.1 - Network plot (one applicant)
Each **node** is a feature; its size and colour show the feature's own effect (order 1). Each **edge** between two nodes is a pairwise interaction (order 2): thicker = stronger, red = positive (together they push towards "default"), blue = negative.
Pairs weaker than `NET_THRESHOLD` are not drawn (they would be too faint to see). Lower it to show more edges.

In [ ]:
fig, ax = shapiq.plot.network_plot(
    interactions[0], feature_names=feature_names, show=False,
    size_factor=3.0, node_size_scaling=0.6,   # thicker edges and smaller nodes
    draw_threshold=NET_THRESHOLD,                      # hide pairs with |value| below this
)
fig.patch.set_facecolor("#c8c8c8")            # grey background: pale nodes and thin edges stay visible
ax.set_facecolor("#c8c8c8")
plt.show()

### Plot 2.2 - Force plots: features only vs features + interactions (same applicant)
This is the kind of plot from your class. **Top:** plain Shapley values, one bar per feature. **Bottom:** the same prediction split into single features *and* feature pairs (written `A x B`).
Both start at the base value and end at the same prediction. Red pushes the prediction up (towards default), blue pushes it down.

In [ ]:
# Top: plain Shapley values (one value per feature, order 1 only) for the same applicant.
explainer_sv = shapiq.TabularExplainer(
    model=predict_positive,
    data=shap.sample(X_train, N_BACKGROUND, random_state=SEED).values,
    index="SV", max_order=1, random_state=SEED)
explainer_sv.explain(rows[0], budget=IQ_BUDGET).plot_force(feature_names=feature_names)

# Bottom: Shapley interaction values (single features + pairs). Same prediction, split into more parts.
interactions[0].plot_force(feature_names=feature_names)

### Plot 2.3 - Heat map: average absolute effect over the explained rows (the `N_TOP` most important features)
Diagonal = effect of each single feature, off-diagonal = effect of each pair (symmetric). Brighter = that feature (or pair) matters more on average.

In [ ]:
# Average absolute single-feature (diagonal) and pair (off-diagonal) values over the explained rows
n = len(feature_names)
heat = np.zeros((n, n))
for iv in interactions:
    heat += np.abs(iv.get_n_order_values(2))                 # pairs (n x n, symmetric)
    heat += np.diag(np.abs(iv.get_n_order_values(1)))        # single features on the diagonal
heat /= len(interactions)

# Keep only the N_TOP most important features (largest diagonal value)
top = np.argsort(np.diag(heat))[::-1][:N_TOP]
heat = heat[np.ix_(top, top)]
names = [feature_names[i] for i in top]
k = len(names)

plt.figure(figsize=(8, 7))
plt.imshow(heat, cmap="Reds")
plt.colorbar(label="mean |value|")
plt.xticks(range(k), names, rotation=45, ha="right")
plt.yticks(range(k), names)
for i in range(k):
    for j in range(k):
        plt.text(j, i, f"{heat[i, j]:.3f}", ha="center", va="center", fontsize=8)
plt.title("Interaction heat map (diagonal = single features)")
plt.tight_layout()
plt.show()

## Part 3: Counterfactual examples with DiCE

A counterfactual answers: *"what is the smallest change to this loan that flips the prediction from "default" to "fully paid"?"*
We use [DiCE](https://github.com/interpretml/DiCE) (`dice-ml`), a standard library that works with any model that has `predict_proba`.
TabPFN has no gradients, so we use DiCE's `genetic` method (it evolves candidate changes and keeps the ones that flip the prediction while staying close to the original).

For each loan the model predicts to default, we show the original row followed by its counterfactuals (a dash = same as the original, only changed features are shown).
We only let DiCE change a few features a borrower could plausibly change (`can_change`); everything else stays fixed. Features that depend on each other (e.g. `installment` depends on `loan_amnt`) are not kept consistent by DiCE.

In [ ]:
# DiCE calls predict_proba many times, so wrap TabPFN to predict in chunks (keeps GPU memory small)
class ChunkedTabPFN:
    def __init__(self, model):
        self.model = model
    def predict_proba(self, data):
        data = data[X_train.columns].astype(X_train.dtypes.to_dict())   # DiCE may reorder columns / change dtypes
        parts = [self.model.predict_proba(data.iloc[i:i + PRED_BATCH]) for i in range(0, len(data), PRED_BATCH)]
        return np.vstack(parts)
    def predict(self, data):
        return self.predict_proba(data).argmax(1)

# Tell DiCE about the data: which columns are numbers and which column is the target
train_df = X_train.assign(**{TARGET: y_train})
continuous = [c for c in X_train.columns if c not in cat_cols]
dice_data = dice_ml.Data(dataframe=train_df, continuous_features=continuous, outcome_name=TARGET)
dice_model = dice_ml.Model(model=ChunkedTabPFN(model), backend="sklearn")
dice = dice_ml.Dice(dice_data, dice_model, method="genetic")

In [ ]:
# Pick test rows the model assigns to class FLIP_FROM; we look for the smallest change that flips them
# (closest to the 0.5 decision boundary first: they are the easiest to flip)
flip_rows = np.where(pred == FLIP_FROM)[0]
flip_rows = flip_rows[np.argsort(abs(proba[flip_rows, 1] - 0.5))]
to_flip = X_test.iloc[flip_rows[:N_CF_APPLICANTS]]

# Only these features may change
can_change = ["loan_amnt", "term_months", "dti", "revol_util", "revol_bal", "int_rate"]

def run_dice():
    try:
        result = dice.generate_counterfactuals(
            to_flip, total_CFs=N_CFS_EACH, desired_class="opposite", features_to_vary=can_change)
    except Exception as e:      # DiCE raises an error when it finds no counterfactual for any row
        print("DiCE found no counterfactuals:", e)
        return {}
    tables = {}
    for idx, cf in zip(to_flip.index, result.cf_examples_list):
        if cf.final_cfs_df is None or len(cf.final_cfs_df) == 0:
            continue                                          # no counterfactual found for this applicant
        original = cf.test_instance_df.drop(columns=TARGET, errors="ignore").reset_index(drop=True)
        cfs = cf.final_cfs_df.drop(columns=TARGET, errors="ignore").reset_index(drop=True)
        tables[idx] = (original, cfs)
    return tables

cf_tables = cached(f"dice_rows{N_CF_APPLICANTS}_cfs{N_CFS_EACH}", run_dice)

In [ ]:
# For each applicant: the original row first, then the counterfactuals.
# In the counterfactual rows a dash means "same as the original"; only the changed features are shown.
for idx, (original, cfs) in cf_tables.items():
    original = original[X_train.columns].astype(float)
    cfs = cfs[X_train.columns].astype(float)
    changed = ~np.isclose(cfs.values, original.values, equal_nan=True)           # True where the counterfactual differs
    shown_cfs = cfs.round(2).astype(object).where(changed, "-")

    table = pd.concat([original.round(2).astype(object), shown_cfs])
    table.index = ["original"] + [f"counterfactual {i + 1}" for i in range(len(cfs))]
    for col in cat_cols:                                         # show category names instead of integer codes
        table.loc["original", col] = category_names[col][int(original[col].iloc[0])]
    table[f"P({TARGET})"] = predict_positive(pd.concat([original, cfs]).values).round(2)

    print(f"Applicant: test row {idx}")
    display(table)